# 03 - Inference: packaged model, MLflow registry and the REST API

In [1]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("CARDIORISK_ROOT", str(ROOT))
print("project root:", ROOT)

project root: /home/claude/heart-disease-mlops


## a) The packaged artefact used by the Docker image (`models/model.joblib`)

In [2]:
import json, joblib, pandas as pd, numpy as np
from cardiorisk import config
model = joblib.load(config.MODEL_PATH)
meta = json.loads(config.METADATA_PATH.read_text())
patients = pd.DataFrame([json.loads((ROOT / "samples" / f).read_text()) for f in ["high_risk.json","low_risk.json","partial_input.json"]],
                        columns=config.INPUT_FEATURES).astype(float)
proba = model.predict_proba(patients)[:, 1]
patients.assign(probability=proba.round(4), prediction=(proba >= meta["decision_threshold"]).astype(int))

,age,trestbps,chol,thalach,oldpeak,sex,cp,fbs,restecg,exang,slope,ca,thal,probability,prediction
0,58.0,140.0,260.0,120.0,2.4,1.0,4.0,0.0,2.0,1.0,2.0,2.0,7.0,0.9808,1
1,37.0,120.0,200.0,185.0,0.0,0.0,3.0,0.0,0.0,0.0,1.0,0.0,3.0,0.0162,0
2,61.0,150.0,NaN,115.0,1.8,1.0,4.0,NaN,NaN,1.0,NaN,NaN,NaN,0.9403,1


## b) Same model from the MLflow Model Registry (`models:/heart-disease-classifier@champion`)

In [3]:
import mlflow
mlflow.set_tracking_uri(f"sqlite:///{ROOT / 'mlflow.db'}")
pyfunc = mlflow.pyfunc.load_model("models:/heart-disease-classifier@champion")
print(pyfunc.metadata.run_id)
# the sklearn flavour gives probabilities; must match the joblib artefact exactly
sk = mlflow.sklearn.load_model("models:/heart-disease-classifier@champion")
assert np.allclose(sk.predict_proba(patients)[:, 1], proba)
print("registry model == packaged model")

786c9c5ab40d4147bdc7e39b9d10f947
registry model == packaged model


## c) Through the FastAPI service (in-process test client - no server needed)

In [4]:
from fastapi.testclient import TestClient
from cardiorisk.serving.app import app
with TestClient(app) as client:
    for f in ["high_risk.json", "low_risk.json", "partial_input.json", "invalid.json"]:
        r = client.post("/predict", json=json.loads((ROOT / "samples" / f).read_text()))
        print(f, r.status_code, r.json())

{"ts": "2026-10-10T15:11:39.541+00:00", "level": "INFO", "logger": "cardiorisk.serving.model_store", "msg": "Loading model from /home/claude/heart-disease-mlops/models/model.joblib"}


{"ts": "2026-10-10T15:11:39.550+00:00", "level": "INFO", "logger": "cardiorisk.api", "msg": "model_loaded", "model_version": "1.0.0+327a95bf", "model_family": "logistic_regression", "threshold": 0.39}


{"ts": "2026-10-10T15:11:39.564+00:00", "level": "INFO", "logger": "cardiorisk.api", "msg": "prediction", "request_id": "f31804c0e279", "prediction": 1, "probability": 0.9808, "risk_band": "high", "missing": []}


{"ts": "2026-10-10T15:11:39.565+00:00", "level": "INFO", "logger": "cardiorisk.api", "msg": "request", "request_id": "f31804c0e279", "method": "POST", "path": "/predict", "status": 200, "latency_ms": 11.37, "client": "testclient"}


{"ts": "2026-10-10T15:11:39.566+00:00", "level": "INFO", "logger": "httpx", "msg": "HTTP Request: POST http://testserver/predict \"HTTP/1.1 200 OK\""}


...

{"ts": "2026-10-10T15:11:39.585+00:00", "level": "INFO", "logger": "httpx", "msg": "HTTP Request: POST http://testserver/predict \"HTTP/1.1 200 OK\""}


partial_input.json 200 {'prediction': 1, 'label': 'heart disease risk', 'probability': 0.9403, 'confidence': 0.9403, 'risk_band': 'high', 'threshold': 0.39, 'missing_features': ['chol', 'fbs', 'restecg', 'slope', 'ca', 'thal'], 'request_id': '6cf7cfe3d322', 'model_version': '1.0.0+327a95bf'}
{"ts": "2026-10-10T15:11:39.588+00:00", "level": "INFO", "logger": "cardiorisk.api", "msg": "validation_error", "request_id": "033ae4181833", "errors": [{"field": "age", "error": "Input should be greater than or equal to 18"}, {"field": "cp", "error": "Input should be 1, 2, 3 or 4"}, {"field": "chol", "error": "Input should be a valid number, unable to parse string as a number"}]}


{"ts": "2026-10-10T15:11:39.588+00:00", "level": "INFO", "logger": "cardiorisk.api", "msg": "request", "request_id": "033ae4181833", "method": "POST", "path": "/predict", "status": 422, "latency_ms": 0.94, "client": "testclient"}


{"ts": "2026-10-10T15:11:39.589+00:00", "level": "INFO", "logger": "httpx", "msg": "HTTP Request: POST http://testserver/predict \"HTTP/1.1 422 Unprocessable Entity\""}


invalid.json 422 {'detail': [{'field': 'age', 'error': 'Input should be greater than or equal to 18'}, {'field': 'cp', 'error': 'Input should be 1, 2, 3 or 4'}, {'field': 'chol', 'error': 'Input should be a valid number, unable to parse string as a number'}]}
